In [2]:
import numpy as np
import csv
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Cargar FAISS
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, model_kwargs={"device": "cpu"})
vs = FAISS.load_local("data/faiss", embeddings, allow_dangerous_deserialization=True)

# Extraer vectores
vectors = []
metadatas = []

# Recorremos cada documento guardado en FAISS
for doc in vs.docstore._dict.values():
    vec = vs.embed_documents([doc.page_content])[0]  # Obtener embedding del chunk
    vectors.append(vec)
    metadatas.append(doc.metadata)

RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open data/faiss/index.faiss for reading: No such file or directory

In [ ]:
with open("data/vectors.tsv", "w", newline="", encoding="utf-8") as f:
    tsv_writer = csv.writer(f, delimiter="\t")
    for vec in vectors:
        tsv_writer.writerow(vec)

In [ ]:
# Extraer las columnas que quieras mostrar
columns = ["titulo", "tipo"]  # o agrega "conceptos", "chunk_id", etc.

with open("data/metadata.tsv", "w", newline="", encoding="utf-8") as f:
    tsv_writer = csv.writer(f, delimiter="\t")
    
    # Encabezado
    tsv_writer.writerow(columns)
    
    # Filas
    for meta in metadatas:
        tsv_writer.writerow([",".join(meta[c]) if isinstance(meta[c], list) else meta[c] for c in columns])